In [ ]:
import numpy as np
import pickle
from scipy.sparse import load_npz, csr_matrix
import pandas as pd
from datetime import datetime

print("=" * 40)
print("CF INFERENCE PIPELINE (ALS)")
print("=" * 40)

# Load ALL trained artifacts

print("\n" + "-" * 40)
print("Loading ALS model and artifacts...")

# Load ALS model
with open('../models/cf/als_model.pkl', 'rb') as f:
    als_model = pickle.load(f)

# Load mappers
with open('../models/cf/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('../models/cf/user_inv_mapper.pkl', 'rb') as f:
    user_inv_mapper = pickle.load(f)

with open('../models/cf/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

with open('../models/cf/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)

# Load training matrix needed for user history
C_train = load_npz('../models/cf/als_confidence_matrix_temporal.npz')  # users × items

print(f"ALS model loaded:")
print(f"  User factors: {als_model.user_factors.shape} (users × K)")
print(f"  Item factors: {als_model.item_factors.shape} (items × K)")
print(f"  Users in mapper: {len(user_mapper):,}")
print(f"  Movies in mapper: {len(movie_mapper):,}")

CF INFERENCE PIPELINE (ALS)

----------------------------------------
Loading ALS model and artifacts...
ALS model loaded:
  User factors: (270882, 100) (users × K)
  Item factors: (45423, 100) (items × K)
  Users in mapper: 270,882
  Movies in mapper: 45,423


In [ ]:
# ALS Inference Functions

def get_user_history(user_id: int) -> list:
    """Get movies user has interacted with"""
    if user_id not in user_mapper:
        return []

    user_idx = user_mapper[user_id]
    user_column = C_train[:, user_idx].tocsc()
    item_indices = user_column.indices
    return [movie_inv_mapper[idx] for idx in item_indices if idx in movie_inv_mapper]

# Calculate global movie popularity once
print("Calculating global movie popularity...")
movie_popularity = {}
for movie_id, idx in movie_mapper.items():
    # For items×users matrix, popularity = number of users who interacted
    popularity = C_train[idx].nnz if idx < C_train.shape[0] else 0
    movie_popularity[movie_id] = popularity

max_pop = max(movie_popularity.values()) if movie_popularity else 1
movie_popularity_norm = {mid: pop/max_pop for mid, pop in movie_popularity.items()}
print(f"Popularity calculated for {len(movie_popularity_norm)} movies")


def predict_als_scores_vectorized(user_id: int, candidate_movies: list) -> dict:
    """
    Predict ALS scores for a user on a list of candidate movies
    Uses vectorized operations for efficiency
    """
    # Cold user handling with popularity fallback
    if user_id not in user_mapper:
        # return {movie_id: movie_popularity_norm.get(movie_id, 0.01)
        #         for movie_id in candidate_movies}
        return {movie_id: 0.0 for movie_id in candidate_movies}

    user_idx = user_mapper[user_id]
    user_vector = als_model.user_factors[user_idx]  # Shape: (K,)

    # Separate known and unknown movies
    known_movies = []
    known_indices = []
    unknown_movies = []

    for movie_id in candidate_movies:
        if movie_id in movie_mapper:
            known_movies.append(movie_id)
            known_indices.append(movie_mapper[movie_id])
        else:
            unknown_movies.append(movie_id)

    scores = {}

    # Vectorized scoring for known movies
    if known_indices:
        item_vectors = als_model.item_factors[known_indices]  # Shape: (n_known, K)
        all_scores = np.dot(item_vectors, user_vector)  # Single operation
        for i, movie_id in enumerate(known_movies):
            scores[movie_id] = float(all_scores[i])

    # For unknown movies, return 0.0 (not popularity)
    for movie_id in unknown_movies:
    #     scores[movie_id] = movie_popularity_norm.get(movie_id, 0.01)
        scores[movie_id] = 0.0

    return scores


def normalize_scores(scores_dict: dict) -> dict:
    """
    Min-max normalize scores to [0, 1] range
    Critical for combining with CBF cosine similarities
    """
    if not scores_dict:
        return {}

    values = np.array(list(scores_dict.values()))

    # Handle edge case where all scores are identical
    if values.max() == values.min():
        return {k: 0.5 for k in scores_dict.keys()}

    normalized = (values - values.min()) / (values.max() - values.min())
    return {k: normalized[i] for i, k in enumerate(scores_dict.keys())}

def get_alpha_als(user_id: int) -> float:
    """
    Determine CF weight based on user history
    ALS is more sensitive to cold start than SVD++
    """
    if user_id is None:
        return 0.0

    history = get_user_history(user_id)
    n_interactions = len(history)

    # Stricter thresholds for ALS (needs more data to be reliable)
    if n_interactions < 5:
        return 0.0      # Pure CBF - ALS vectors are noise
    elif n_interactions < 15:
        return 0.25     # CBF-dominant
    elif n_interactions < 30:
        return 0.5      # Balanced
    else:
        return 0.75     # CF-dominant (but never 1.0)

Calculating global movie popularity...
Popularity calculated for 45423 movies


In [ ]:
# Test the Inference Pipeline

print("\n" + "-" * 40)
print("Testing ALS Inference Pipeline")

# Select test user (using same user 38150 from before)
test_user = 38150
print(f"\nTest User: {test_user}")

# Get user history
history = get_user_history(test_user)
print(f"User has {len(history)} interactions in history")
print(f"Sample history (first 5): {history[:5]}")

# Calculate alpha for this user
alpha = get_alpha_als(test_user)
print(f"Alpha value: {alpha:.2f} (CF weight)")

# Simulate candidate movies (in reality, these come from CBF)
# Using some random movies + some from user's history to test filtering
candidate_movies = [10451, 10149, 2759, 11010, 235, 10997, 11448, 695, 11382, 9549,
                    1600, 11860, 807, 623]  # Mix of recommendations + history

# Get ALS scores
raw_scores = predict_als_scores_vectorized(test_user, candidate_movies)
print(f"\nRaw ALS scores (sample):")
for movie_id in candidate_movies[:5]:
    print(f"  Movie {movie_id}: {raw_scores[movie_id]:.4f}")

# Normalize scores
norm_scores = normalize_scores(raw_scores)
print(f"\nNormalized scores [0-1] (sample):")
for movie_id in candidate_movies[:5]:
    print(f"  Movie {movie_id}: {norm_scores[movie_id]:.4f}")



----------------------------------------
Testing ALS Inference Pipeline

Test User: 38150
User has 115 interactions in history
Sample history (first 5): [862, 4584, 8012, 451, 63]
Alpha value: 0.75 (CF weight)

Raw ALS scores (sample):
  Movie 10451: 1.2987
  Movie 10149: 1.2379
  Movie 2759: 1.2134
  Movie 11010: 1.1810
  Movie 235: 1.0673

Normalized scores [0-1] (sample):
  Movie 10451: 1.0000
  Movie 10149: 0.9309
  Movie 2759: 0.9031
  Movie 11010: 0.8663
  Movie 235: 0.7372


In [ ]:
# Cold Start Test

print("\n" + "-" * 40)
print("Testing Cold Start (New User)")

new_user = 999999999  # Non-existent user
new_user_alpha = get_alpha_als(new_user)
new_user_scores = predict_als_scores_vectorized(new_user, candidate_movies[:5])

print(f"New user alpha: {new_user_alpha}")
print(f"New user scores (should all be 0.0):")
for movie_id, score in new_user_scores.items():
    print(f"  Movie {movie_id}: {score:.4f}")


----------------------------------------
Testing Cold Start (New User)
New user alpha: 0.0
New user scores (should all be 0.0):
  Movie 10451: 0.0000
  Movie 10149: 0.0000
  Movie 2759: 0.0000
  Movie 11010: 0.0000
  Movie 235: 0.0000


In [ ]:
# Performance Test

print("\n" + "-" * 40)
print("Performance Test (50 candidates)")


import time

# Simulate 50 candidate movies
test_candidates = list(movie_mapper.keys())[:50]

# start = time.time()
start = time.perf_counter()
scores = predict_als_scores_vectorized(test_user, test_candidates)
# end = time.time()
end = time.perf_counter()

print(f"Time to score 50 candidates: {(end-start)*1000:.2f} ms")
print(f"Average per candidate: {(end-start)*1000/50:.2f} ms")


----------------------------------------
Performance Test (50 candidates)
Time to score 50 candidates: 0.83 ms
Average per candidate: 0.02 ms


In [ ]:
# Save Inference Engine Components

print("\n" + "-" * 40)
print("Saving inference-ready components")

# Create a lightweight inference package
inference_artifacts = {
    'model_type': 'ALS',
    'user_mapper': user_mapper,
    'movie_mapper': movie_mapper,
    'movie_inv_mapper': movie_inv_mapper,
    'n_users': len(user_mapper),
    'n_movies': len(movie_mapper),
    'factors': als_model.user_factors.shape[1],
    'alpha_thresholds': {
        'pure_cbf': 5,
        'cbf_dominant': 15,
        'balanced': 30,
        'cf_dominant': 75
    }
}

with open('../models/cf/cf_inference_config.pkl', 'wb') as f:
    pickle.dump(inference_artifacts, f)
print("✓ Saved: cf_inference_config.pkl")

print("\n" + "=" * 60)
print("COMPLETE - ALS INFERENCE PIPELINE READY")
print("=" * 60)



----------------------------------------
Saving inference-ready components
✓ Saved: cf_inference_config.pkl

COMPLETE - ALS INFERENCE PIPELINE READY


In [ ]:
# CF EVALUATION

import numpy as np
import pandas as pd
import pickle
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.model_selection import train_test_split
import implicit
import mlflow
from datetime import datetime
from tqdm.auto import tqdm
import time
import heapq

print("=" * 60)
print("CF EVALUATION")
print("=" * 60)

# 15.1 Load Original Data (Before Any Matrix Building)

print("\n" + "-" * 40)
print("Loading original ratings data...")

# Load original ratings (with double-centering already applied)
ratings = pd.read_csv('/content/drive/MyDrive/ratings_centered.csv')
print(f"Total ratings: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['tmdbId'].nunique():,}")

# Load mappers (these are just ID mappings, not model-dependent)
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)

print(f"User mapper: {len(user_mapper):,}")
print(f"Movie mapper: {len(movie_mapper):,}")

# Create Train/Test Split

print("\n" + "-" * 40)
print("Creating train/test split (20% per user)")

def create_user_split(ratings_df, test_size=0.2, random_state=42):
    """
    Split ratings by user to prevent leakage
    Each user gets 80% train, 20% test
    """
    np.random.seed(random_state)

    train_indices = []
    test_indices = []

    # Group by user
    for user_id in ratings_df['userId'].unique():
        user_rows = ratings_df[ratings_df['userId'] == user_id].index.tolist()

        # Users with < 5 ratings go entirely to train (can't evaluate)
        if len(user_rows) < 5:
            train_indices.extend(user_rows)
            continue

        # Randomly select test_size% for test
        n_test = max(1, int(len(user_rows) * test_size))
        test_idx = np.random.choice(user_rows, size=n_test, replace=False)

        test_indices.extend(test_idx)
        train_indices.extend([idx for idx in user_rows if idx not in test_idx])

    train_df = ratings_df.loc[train_indices].copy()
    test_df = ratings_df.loc[test_indices].copy()

    print(f"Training set: {len(train_df):,} ratings ({len(train_df)/len(ratings_df)*100:.1f}%)")
    print(f"Test set: {len(test_df):,} ratings ({len(test_df)/len(ratings_df)*100:.1f}%)")
    print(f"Users in train: {train_df['userId'].nunique():,}")
    print(f"Users in test: {test_df['userId'].nunique():,}")

    return train_df, test_df

ratings_train, ratings_test = create_user_split(ratings)

# Build Confidence Matrix

print("\n" + "-" * 40)
print("Building confidence matrix from TRAIN data")

# Map indices
ratings_train['user_idx'] = ratings_train['userId'].map(user_mapper)
ratings_train['item_idx'] = ratings_train['tmdbId'].map(movie_mapper)

# Remove any missing mappings (should be none if data is clean)
ratings_train = ratings_train.dropna(subset=['user_idx', 'item_idx']).copy()
ratings_train['user_idx'] = ratings_train['user_idx'].astype(int)
ratings_train['item_idx'] = ratings_train['item_idx'].astype(int)

# Build confidence scores
if 'implicit_confidence' not in ratings_train.columns:
    # Create it from rating (0.5-5.0 → 0.1-1.0)
    ratings_train['implicit_confidence'] = ratings_train['rating'] / 5.0
alpha = 40
ratings_train['confidence'] = 1 + alpha * ratings_train['implicit_confidence']

print(f"Confidence stats:")
print(f"  Min: {ratings_train['confidence'].min():.2f}")
print(f"  Max: {ratings_train['confidence'].max():.2f}")
print(f"  Mean: {ratings_train['confidence'].mean():.2f}")

# Create sparse matrix (items × users)
n_users = len(user_mapper)
n_items = len(movie_mapper)

C_train = csr_matrix(
    (
        ratings_train['confidence'].astype(np.float32),
        (ratings_train['item_idx'], ratings_train['user_idx'])
    ),
    shape=(n_items, n_users),
    dtype=np.float32
)

print(f"\nTraining matrix shape: {C_train.shape} (items × users)")
print(f"Non-zero entries: {C_train.nnz:,}")
print(f"Density: {C_train.nnz / (n_items * n_users) * 100:.6f}%")

# Train ALS on

print("\n" + "-" * 40)
print("Training ALS model on TRAIN data")

model = implicit.als.AlternatingLeastSquares(
    factors=100,           # K=100 (spec)
    regularization=0.01,    # λ=0.01 (spec)
    iterations=50,          # 50 iterations (spec)
    alpha=1.0,              # implicit library handles α internally
    random_state=42,
    num_threads=0,
    show_progress=True,
    use_gpu=False
)

start_time = time.time()
model.fit(C_train)
# C_for_training = C.T.tocsr()
train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")
print(f"User factors: {model.user_factors.shape}")
print(f"Item factors: {model.item_factors.shape}")

# Save model locally
eval_model_path = "/content/drive/MyDrive/als_model_eval.pkl"
with open(eval_model_path, "wb") as f:
    pickle.dump(model, f)

# RANKING METRICS ONLY

print("\n" + "-" * 40)
print("Computing ranking metrics on TEST data")

def precision_at_k(recommended, relevant, k):
    """Precision@K"""
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    """Recall@K"""
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def ndcg_at_k(recommended, relevant, k):
    """NDCG@K"""
    if len(recommended) > k:
        recommended = recommended[:k]

    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / ideal if ideal > 0 else 0

def map_at_k(recommended, relevant, k):
    """Mean Average Precision@K"""
    if len(recommended) > k:
        recommended = recommended[:k]

    ap = 0
    hits = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k) if len(relevant) > 0 else 0

# Select test users with enough interactions
test_user_counts = ratings_test['userId'].value_counts()
test_users = test_user_counts[test_user_counts >= 3].index.tolist()
print(f"Evaluating on {len(test_users)} test users (with ≥3 test interactions)")

# Get training seen items per user
user_seen_items = {}
for user_id in ratings_train['userId'].unique():
    user_seen = ratings_train[ratings_train['userId'] == user_id]['tmdbId'].tolist()
    user_seen_items[user_id] = set(user_seen)

metrics = {'p@5': [], 'p@10': [], 'r@5': [], 'r@10': [], 'ndcg@10': [], 'map@10': []}



# Build lookup dictionaries ONCE (O(N))
user_test_dict = {}
for user_id, group in ratings_test.groupby('userId'):
    user_test_dict[user_id] = group

user_seen_dict = {}
for user_id, group in ratings_train.groupby('userId'):
    user_seen_dict[user_id] = set(group['tmdbId'].tolist())

# Then in loop:
user_test = user_test_dict.get(user_id, pd.DataFrame())
seen_movies = user_seen_dict.get(user_id, set())

for user_id in test_users:
    # Get user's test items (relevant if rating ≥ 3.5)
    user_test = ratings_test[ratings_test['userId'] == user_id]
    relevant_movies = user_test[user_test['rating'] >= 3.5]['tmdbId'].tolist()

    if len(relevant_movies) == 0:
        continue

    # Get user's seen items from training
    seen_movies = user_seen_items.get(user_id, set())

    # Generate recommendations
    if user_id in user_mapper:
        user_idx = user_mapper[user_id]
        user_vector = model.user_factors[user_idx]

        # Score all items (vectorized)
        all_scores = np.dot(model.item_factors, user_vector)

        # Mask seen items
        seen_mask = np.zeros(len(movie_mapper), dtype=bool)
        for movie_id in seen_movies:
            if movie_id in movie_mapper:
                seen_mask[movie_mapper[movie_id]] = True

        # Vectorized operation
        scores_copy = all_scores.copy()
        scores_copy[seen_mask] = -np.inf

        # Get top 20 recommendations
        top_k = 20
        # top_indices = np.argsort(scores_copy)[-top_k:][::-1]
        top_indices = heapq.nlargest(top_k, range(len(all_scores)), all_scores.__getitem__)
        recommendations = [movie_inv_mapper[idx] for idx in top_indices
                          if idx in movie_inv_mapper]
    else:
        continue

    # Calculate metrics
    metrics['p@5'].append(precision_at_k(recommendations, relevant_movies, 5))
    metrics['p@10'].append(precision_at_k(recommendations, relevant_movies, 10))
    metrics['r@5'].append(recall_at_k(recommendations, relevant_movies, 5))
    metrics['r@10'].append(recall_at_k(recommendations, relevant_movies, 10))
    metrics['ndcg@10'].append(ndcg_at_k(recommendations, relevant_movies, 10))
    metrics['map@10'].append(map_at_k(recommendations, relevant_movies, 10))

# Print results
print("\n" + "=" * 40)
print("FINAL EVALUATION RESULTS")
print("=" * 40)
print(f"Precision@5:  {np.mean(metrics['p@5']):.4f} ± {np.std(metrics['p@5']):.4f}")
print(f"Precision@10: {np.mean(metrics['p@10']):.4f} ± {np.std(metrics['p@10']):.4f}")
print(f"Recall@5:     {np.mean(metrics['r@5']):.4f} ± {np.std(metrics['r@5']):.4f}")
print(f"Recall@10:    {np.mean(metrics['r@10']):.4f} ± {np.std(metrics['r@10']):.4f}")
print(f"NDCG@10:      {np.mean(metrics['ndcg@10']):.4f} ± {np.std(metrics['ndcg@10']):.4f}")
print(f"MAP@10:       {np.mean(metrics['map@10']):.4f} ± {np.std(metrics['map@10']):.4f}")

# MLflow Tracking

print("\n" + "-" * 40)
print("Logging to MLflow")

mlflow_dir = "/content/drive/MyDrive/mlflow_data"
mlflow.set_tracking_uri(f"file:{mlflow_dir}")
with mlflow.start_run(run_name=f"als_evaluation_clean_{datetime.now().strftime('%Y%m%d_%H%M')}"):

    # Parameters
    mlflow.log_param("algorithm", "ALS")
    mlflow.log_param("n_factors", 100)
    mlflow.log_param("regularization", 0.01)
    mlflow.log_param("iterations", 50)
    mlflow.log_param("alpha", 40)
    mlflow.log_param("train_size", len(ratings_train))
    mlflow.log_param("test_size", len(ratings_test))
    mlflow.log_param("test_users", len(test_users))

    # Metrics (ranking only)
    mlflow.log_metric("precision@5", np.mean(metrics['p@5']))
    mlflow.log_metric("precision@10", np.mean(metrics['p@10']))
    mlflow.log_metric("recall@5", np.mean(metrics['r@5']))
    mlflow.log_metric("recall@10", np.mean(metrics['r@10']))
    mlflow.log_metric("ndcg@10", np.mean(metrics['ndcg@10']))
    mlflow.log_metric("map@10", np.mean(metrics['map@10']))
    mlflow.log_metric("training_time_seconds", train_time)

    # Artifacts
    mlflow.log_artifact("/content/drive/MyDrive/user_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/movie_mapper.pkl")
    mlflow.log_artifact(eval_model_path, artifact_path="evaluation_model")

    # Tags
    mlflow.set_tag("model_type", "collaborative_filtering")
    mlflow.set_tag("algorithm", "ALS")
    mlflow.set_tag("evaluation_protocol", "clean_train_test_split")
    mlflow.set_tag("leakage_prevented", "true")

print("\n" + "=" * 60)
print("COMPLETE - CLEAN EVALUATION DONE")
print("=" * 60)

CF EVALUATION

----------------------------------------
Loading original ratings data...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install mlflow
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Get

In [ ]:
"""
NOTE: This evaluation focuses on ALS as the primary collaborative filtering algorithm.
Per Section 13 of the spec, ALS is a valid production alternative to SVD++.
KNN-CF and SVD++ baselines are omitted as the project scope was adjusted to
prioritize ALS implementation. All 9 metrics (P@5,10,20; R@5,10,20; NDCG@10,20; MAP@10,20)
are computed for ALS as specified in Section 15.2.
"""

import numpy as np
import pandas as pd
import pickle
from scipy.sparse import csr_matrix
import implicit
import mlflow
from datetime import datetime
from tqdm.auto import tqdm
import time
import os
import gc

print("=" * 60)
print("CF EVALUATION - PRODUCTION OPTIMIZED")
print("=" * 60)


print("\n" + "-" * 40)
print("Loading original ratings data...")

ratings = pd.read_csv('/content/drive/MyDrive/ratings_centered.csv')
print(f"Total ratings: {len(ratings):,}")
print(f"Users: {ratings['userId'].nunique():,}")
print(f"Movies: {ratings['tmdbId'].nunique():,}")

# Load mappers
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)

with open('/content/drive/MyDrive/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)

print(f"User mapper: {len(user_mapper):,}")
print(f"Movie mapper: {len(movie_mapper):,}")


CF EVALUATION - PRODUCTION OPTIMIZED

----------------------------------------
Loading original ratings data...
Total ratings: 25,981,438
Users: 270,882
Movies: 44,702
User mapper: 270,882
Movie mapper: 45,423


In [ ]:
print("\n" + "-" * 40)
print("Creating train/test split (20% per user) - VECTORIZED")

def create_user_split_vectorized(ratings_df, test_size=0.2, random_state=42):
    """
    Vectorized split by user to prevent leakage.
    Instead of looping through users, we use group-based shuffling.
    This runs in O(N) time instead of O(U×N).
    """
    print("Vectorizing split logic...")
    np.random.seed(random_state)

    df = ratings_df.copy()
    df['random_val'] = np.random.rand(len(df))
    df['user_rank'] = df.groupby('userId')['random_val'].rank(pct=True)

    user_counts = df.groupby('userId')['userId'].transform('count')
    test_mask = (user_counts >= 5) & (df['user_rank'] <= test_size)

    train_df = df[~test_mask].drop(columns=['random_val', 'user_rank']).copy()
    test_df = df[test_mask].drop(columns=['random_val', 'user_rank']).copy()

    # Clean up
    del df
    gc.collect()

    print(f"Training set: {len(train_df):,} ratings ({len(train_df)/len(ratings_df)*100:.1f}%)")
    print(f"Test set: {len(test_df):,} ratings ({len(test_df)/len(ratings_df)*100:.1f}%)")
    print(f"Users in train: {train_df['userId'].nunique():,}")
    print(f"Users in test: {test_df['userId'].nunique():,}")

    return train_df, test_df

# Use the vectorized version
ratings_train, ratings_test = create_user_split_vectorized(ratings)

# Free memory
del ratings
gc.collect()



----------------------------------------
Creating train/test split (20% per user) - VECTORIZED
Vectorizing split logic...
Training set: 20,880,997 ratings (80.4%)
Test set: 5,100,441 ratings (19.6%)
Users in train: 270,882
Users in test: 256,051


0

In [ ]:
print("\n" + "-" * 40)
print("Building confidence matrix from TRAIN data")

# Map indices
ratings_train = ratings_train.copy()
ratings_train['user_idx'] = ratings_train['userId'].map(user_mapper)
ratings_train['item_idx'] = ratings_train['tmdbId'].map(movie_mapper)

# Remove missing
ratings_train = ratings_train.dropna(subset=['user_idx', 'item_idx'])
ratings_train['user_idx'] = ratings_train['user_idx'].astype(int)
ratings_train['item_idx'] = ratings_train['item_idx'].astype(int)

# confidence calculation
ratings_train['implicit_confidence'] = ratings_train['rating'] / 5.0
alpha = 40
ratings_train['confidence'] = 1 + alpha * ratings_train['implicit_confidence']

print(f"Confidence stats:")
print(f"  Min: {ratings_train['confidence'].min():.2f}")
print(f"  Max: {ratings_train['confidence'].max():.2f}")
print(f"  Mean: {ratings_train['confidence'].mean():.2f}")

# Create sparse matrix (items × users)
n_users = len(user_mapper)
n_items = len(movie_mapper)

C_items_users = csr_matrix(
    (
        ratings_train['confidence'].astype(np.float32),
        (ratings_train['item_idx'], ratings_train['user_idx'])
    ),
    shape=(n_items, n_users),
    dtype=np.float32
)

print(f"\nItems×Users matrix shape: {C_items_users.shape}")
print(f"Non-zero entries: {C_items_users.nnz:,}")

# Free memory
del ratings_train
gc.collect()


----------------------------------------
Building confidence matrix from TRAIN data
Confidence stats:
  Min: 5.00
  Max: 41.00
  Mean: 29.23

Items×Users matrix shape: (45423, 270882)
Non-zero entries: 20,880,996


0

In [ ]:
print("\n" + "-" * 40)
print("Converting to User-Item matrix for ALS...")

C_users_items = C_items_users.T.tocsr()
print(f"Users×Items matrix shape: {C_users_items.shape}")
print(f"Non-zero entries: {C_users_items.nnz:,}")


----------------------------------------
Converting to User-Item matrix for ALS...
Users×Items matrix shape: (270882, 45423)
Non-zero entries: 20,880,996


In [ ]:
print("\n" + "-" * 40)
print("Training ALS model on TRAIN data...")

model = implicit.als.AlternatingLeastSquares(
    factors=100,
    regularization=0.01,
    iterations=50,
    random_state=42,
    num_threads=0,          # Use all CPU cores
    use_gpu=False
)

start_time = time.time()
model.fit(C_users_items, show_progress=True)
train_time = time.time() - start_time

print(f"Training completed in {train_time:.2f} seconds")
print(f"User factors: {model.user_factors.shape}")
print(f"Item factors: {model.item_factors.shape}")

# Save model locally
eval_model_path = "/content/drive/MyDrive/als_model_eval.pkl"
with open(eval_model_path, "wb") as f:
    pickle.dump(model, f)


----------------------------------------
Training ALS model on TRAIN data...


/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]

Training completed in 1008.57 seconds
User factors: (270882, 100)
Item factors: (45423, 100)


In [ ]:
print("\n" + "-" * 40)
print("Building lookup dictionaries...")

# Test items by user
user_test_dict = {}
for user_id, group in tqdm(ratings_test.groupby('userId'), desc="Test dict"):
    user_test_dict[user_id] = group

# Create user_inv_mapper if not exists
user_inv_mapper = {idx: uid for uid, idx in user_mapper.items()}

# Seen items from training - using item indices for faster access
user_seen_indices = {}
# We need to rebuild training indices from C_users_items
print("Building seen items indices...")
for user_idx in tqdm(range(C_users_items.shape[0]), desc="Processing users"):
    if user_idx in user_inv_mapper:  # User_inv_mapper
        user_id = user_inv_mapper[user_idx]
        # Get non-zero item indices for this user
        item_indices = C_users_items[user_idx].indices
        user_seen_indices[user_id] = item_indices

# Relevant items (rating ≥ 3.5) for each user in test
user_relevant_dict = {}
for user_id, group in tqdm(user_test_dict.items(), desc="Relevant dict"):
    relevant = group[group['rating'] >= 3.5]['tmdbId'].tolist()
    if relevant:
        user_relevant_dict[user_id] = relevant



----------------------------------------
Building lookup dictionaries...


Test dict:   0%|          | 0/256051 [00:00<?, ?it/s]

Building seen items indices...


Processing users:   0%|          | 0/270882 [00:00<?, ?it/s]

Relevant dict:   0%|          | 0/256051 [00:00<?, ?it/s]

In [ ]:
def precision_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def ndcg_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]

    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / ideal if ideal > 0 else 0

def map_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]

    ap = 0
    hits = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k) if len(relevant) > 0 else 0

In [ ]:
print("\n" + "-" * 40)
print("Running PRODUCTION evaluation with model.recommend()...")

test_users = list(user_relevant_dict.keys())
print(f"Evaluating on {len(test_users)} users with relevant test items")

# Initialize metrics with ALL K values (5, 10, 20)
metrics = {
    'p@5': [], 'p@10': [], 'p@20': [],
    'r@5': [], 'r@10': [], 'r@20': [],
    'ndcg@10': [], 'ndcg@20': [],
    'map@10': [], 'map@20': []
}
N = 20  # Number of recommendations to generate

for user_id in tqdm(test_users, desc="Evaluating"):
    if user_id not in user_mapper:
        continue

    # Get user data
    user_idx = user_mapper[user_id]
    relevant_movies = user_relevant_dict[user_id]

    # Use model.recommend() - MULTI-THREADED
    recommendations = model.recommend(
        user_idx,                       # User index
        C_users_items[user_idx],        # User's current items (sparse row)
        N=N,                            # Number to return
        filter_already_liked_items=True, # Auto-filters seen items
        items=None,                      # All items
        recalculate_user=True            # Recalculate from factors
    )

    # Convert indices to movie IDs
    recommended_indices = recommendations[0]
    recommended_movies = [movie_inv_mapper[idx] for idx in recommended_indices if idx in movie_inv_mapper]

    # Calculate metrics for ALL K values
    metrics['p@5'].append(precision_at_k(recommended_movies, relevant_movies, 5))
    metrics['p@10'].append(precision_at_k(recommended_movies, relevant_movies, 10))
    metrics['p@20'].append(precision_at_k(recommended_movies, relevant_movies, 20))

    metrics['r@5'].append(recall_at_k(recommended_movies, relevant_movies, 5))
    metrics['r@10'].append(recall_at_k(recommended_movies, relevant_movies, 10))
    metrics['r@20'].append(recall_at_k(recommended_movies, relevant_movies, 20))

    metrics['ndcg@10'].append(ndcg_at_k(recommended_movies, relevant_movies, 10))
    metrics['ndcg@20'].append(ndcg_at_k(recommended_movies, relevant_movies, 20))

    metrics['map@10'].append(map_at_k(recommended_movies, relevant_movies, 10))
    metrics['map@20'].append(map_at_k(recommended_movies, relevant_movies, 20))



----------------------------------------
Running PRODUCTION evaluation with model.recommend()...
Evaluating on 236028 users with relevant test items


Evaluating:   0%|          | 0/236028 [00:00<?, ?it/s]

In [ ]:
print("\n" + "=" * 50)
print("FINAL EVALUATION RESULTS - ALL 9 METRICS")
print("=" * 50)
print(f"Precision@5:  {np.mean(metrics['p@5']):.4f} ± {np.std(metrics['p@5']):.4f}")
print(f"Precision@10: {np.mean(metrics['p@10']):.4f} ± {np.std(metrics['p@10']):.4f}")
print(f"Precision@20: {np.mean(metrics['p@20']):.4f} ± {np.std(metrics['p@20']):.4f}")
print(f"Recall@5:     {np.mean(metrics['r@5']):.4f} ± {np.std(metrics['r@5']):.4f}")
print(f"Recall@10:    {np.mean(metrics['r@10']):.4f} ± {np.std(metrics['r@10']):.4f}")
print(f"Recall@20:    {np.mean(metrics['r@20']):.4f} ± {np.std(metrics['r@20']):.4f}")
print(f"NDCG@10:      {np.mean(metrics['ndcg@10']):.4f} ± {np.std(metrics['ndcg@10']):.4f}")
print(f"NDCG@20:      {np.mean(metrics['ndcg@20']):.4f} ± {np.std(metrics['ndcg@20']):.4f}")
print(f"MAP@10:       {np.mean(metrics['map@10']):.4f} ± {np.std(metrics['map@10']):.4f}")
print(f"MAP@20:       {np.mean(metrics['map@20']):.4f} ± {np.std(metrics['map@20']):.4f}")



FINAL EVALUATION RESULTS - ALL 9 METRICS
Precision@5:  0.1712 ± 0.2016
Precision@10: 0.1479 ± 0.1558
Precision@20: 0.1222 ± 0.1246
Recall@5:     0.1445 ± 0.2448
Recall@10:    0.2250 ± 0.2920
Recall@20:    0.3276 ± 0.3274
NDCG@10:      0.2301 ± 0.2382
NDCG@20:      0.2576 ± 0.2317
MAP@10:       0.1369 ± 0.1919
MAP@20:       0.1402 ± 0.1872


In [ ]:
print("\n" + "-" * 40)
print("Creating comparison table (ALS only)")

comparison_data = {
    'Model': ['ALS (implicit)'],
    'P@10': [f"{np.mean(metrics['p@10']):.4f} ± {np.std(metrics['p@10']):.4f}"],
    'R@10': [f"{np.mean(metrics['r@10']):.4f} ± {np.std(metrics['r@10']):.4f}"],
    'NDCG@10': [f"{np.mean(metrics['ndcg@10']):.4f} ± {np.std(metrics['ndcg@10']):.4f}"],
    'Training Time (s)': [f"{train_time:.2f}"]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "=" * 50)
print("MODEL COMPARISON TABLE")
print("=" * 50)
print(comparison_df.to_string(index=False))

# Save table as CSV
table_path = "/content/drive/MyDrive/model_comparison.csv"
comparison_df.to_csv(table_path, index=False)



----------------------------------------
Creating comparison table (ALS only)

MODEL COMPARISON TABLE
         Model            P@10            R@10         NDCG@10 Training Time (s)
ALS (implicit) 0.1479 ± 0.1558 0.2250 ± 0.2920 0.2301 ± 0.2382           1008.57


In [ ]:
# 1. Reset any active runs to clear the state
if mlflow.active_run():
    mlflow.end_run()

print("\n" + "-" * 40)
print("Logging to MLflow (Force SQLite Backend)")

# 2. Define a clean DB path
db_path = "/content/drive/MyDrive/mlflow_data/production_eval.db"
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# 3. SET THE URI (Ensure 'sqlite' prefix is used)
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

# 4. Create and set a NEW experiment name
# This bypasses the search for ID 0 or 1
experiment_name = "ALS_Final_Eval_2026"
try:
    exp_id = mlflow.create_experiment(experiment_name)
except:
    # If it already exists, just get the ID
    exp_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

# 5. Start the run
run_name = f"als_prod_{datetime.now().strftime('%Y%m%d_%H%M')}"
with mlflow.start_run(run_name=run_name):
    print(f"Run started successfully in Experiment: {experiment_name}")

    # Log your 9 metrics here...

    # Parameters
    mlflow.log_param("algorithm", "ALS")
    mlflow.log_param("n_factors", 100)
    mlflow.log_param("regularization", 0.01)
    mlflow.log_param("iterations", 50)
    mlflow.log_param("alpha", 40)
    mlflow.log_param("train_size_interactions", C_users_items.nnz)
    mlflow.log_param("test_size_interactions", len(ratings_test))
    mlflow.log_param("test_users_count", len(test_users))
    mlflow.log_param("optimization", "vectorized_split + model.recommend()")

    # ALL 9 METRICS (3 K values × 3 metrics)
    mlflow.log_metric("precision_5", np.mean(metrics['p@5']))
    mlflow.log_metric("precision_10", np.mean(metrics['p@10']))
    mlflow.log_metric("precision_20", np.mean(metrics['p@20']))

    mlflow.log_metric("recall_5", np.mean(metrics['r@5']))
    mlflow.log_metric("recall_10", np.mean(metrics['r@10']))
    mlflow.log_metric("recall_20", np.mean(metrics['r@20']))

    mlflow.log_metric("ndcg_10", np.mean(metrics['ndcg@10']))
    mlflow.log_metric("ndcg_20", np.mean(metrics['ndcg@20']))

    mlflow.log_metric("map_10", np.mean(metrics['map@10']))
    mlflow.log_metric("map_20", np.mean(metrics['map@20']))

    mlflow.log_metric("training_time_seconds", train_time)

    # Artifacts
    mlflow.log_artifact("/content/drive/MyDrive/user_mapper.pkl")
    mlflow.log_artifact("/content/drive/MyDrive/movie_mapper.pkl")
    mlflow.log_artifact(eval_model_path, artifact_path="evaluation_model")
    mlflow.log_artifact(table_path, artifact_path="comparison_tables")

    # Markdown table for better readability
    mlflow.log_text(
        comparison_df.to_markdown(index=False),
        "comparison_tables/model_comparison.md"
    )

    # Tags
    mlflow.set_tag("model_type", "collaborative_filtering")
    mlflow.set_tag("algorithm", "ALS")
    mlflow.set_tag("evaluation_protocol", "full_metrics")
    mlflow.set_tag("models_evaluated", "ALS_only")
    mlflow.set_tag("split_method", "vectorized")
    mlflow.set_tag("spec_deviation", "KNN-CF_SVD_SVD++_skipped")

print("\n" + "=" * 60)
print("PRODUCTION EVALUATION COMPLETE - ALL 9 METRICS LOGGED")
print("=" * 60)


----------------------------------------
Logging to MLflow (Force SQLite Backend)
Run started successfully in Experiment: ALS_Final_Eval_2026

PRODUCTION EVALUATION COMPLETE - ALL 9 METRICS LOGGED


In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{db_path}")

runs = mlflow.search_runs(experiment_names=["ALS_Final_Eval_2026"])

recent_run = runs.iloc[0]
print(f"Run ID: {recent_run['run_id']}")
print(f"Status: {recent_run['status']}")

# Filter columns to see just the metrics
metric_cols = [c for c in runs.columns if 'metrics.' in c]
print("\nLogged Metrics:")
print(runs[metric_cols].iloc[0])

Run ID: fa717edbb4b3478fbda2f5b10a8ca821
Status: FINISHED

Logged Metrics:
metrics.recall_20                   0.327624
metrics.training_time_seconds    1008.570801
metrics.ndcg_20                     0.257593
metrics.recall_5                    0.144513
metrics.precision_20                0.122167
metrics.recall_10                   0.224977
metrics.ndcg_10                     0.230113
metrics.precision_10                0.147924
metrics.map_20                      0.140200
metrics.map_10                      0.136860
metrics.precision_5                 0.171216
Name: 0, dtype: float64


In [ ]:
%pip install -q dagshub mlflow

In [ ]:
import mlflow
import os
from dotenv import load_dotenv

load_dotenv()

USER_NAME = os.getenv("MLFLOW_TRACKING_USERNAME")
REPO_NAME = os.getenv("MLFLOW_REPO_NAME")
TRACKING_URI = f"https://dagshub.com/{USER_NAME}/{REPO_NAME}.mlflow"
mlflow.set_tracking_uri(TRACKING_URI)

print(f"Authenticated as: {os.getenv('MLFLOW_TRACKING_USERNAME')}")
print(f"Tracking to: {mlflow.get_tracking_uri()}")

In [ ]:
experiment_name = "ALS_Final_Eval_2026"
try:
    exp = mlflow.get_experiment_by_name(experiment_name)
    if exp is None:
        print(f"Creating new experiment: {experiment_name}")
        mlflow.create_experiment(experiment_name)
    mlflow.set_experiment(experiment_name)
except Exception as e:
    print(f"Error setting experiment: {e}")

with mlflow.start_run(run_name=f"als_cloud_eval_{datetime.now().strftime('%H%M')}"):
    # Parameters
    mlflow.log_params({
        "algorithm": "ALS",
        "n_factors": 100,
        "alpha": 40,
        "optimization": "vectorized_split_recommend",
        "deployment_ready": "True"
    })

    # Metrics
    mlflow.log_metrics({
        "precision_k5": np.mean(metrics['p@5']),
        "precision_k10": np.mean(metrics['p@10']),
        "precision_k20": np.mean(metrics['p@20']),
        "recall_k10": np.mean(metrics['r@10']),
        "ndcg_k10": np.mean(metrics['ndcg@10']),
        "map_k10": np.mean(metrics['map@10']),
        "training_time_s": train_time
    })

    mlflow.set_tag("environment", "production_eval")
    print("Success: Run pushed to DagsHub.")

In [2]:
!pip install implicit
!pip install mlflow
!pip install faiss-cpu --no-cache
!pip install --upgrade scikit-learn==1.8.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933265 sha256=93638fefaab441f154d238324b1c09d466b232875808c845c38914f160f03514
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:

import numpy as np
import pandas as pd
import pickle
import faiss
from scipy.sparse import load_npz
import time
import random
from collections import defaultdict
import mlflow
from datetime import datetime
from dotenv import load_dotenv
import os
import gc

print("=" * 60)
print("COMPARATIVE EVALUATION: CBF vs CF vs HYBRID")
print("=" * 60)


print("\n" + "-" * 40)
print("1. LOADING DATA")
print("-" * 40)

# Load original ratings
ratings = pd.read_csv('/content/drive/MyDrive/ratings_centered.csv')
print(f"Total ratings: {len(ratings):,}")

COMPARATIVE EVALUATION: CBF vs CF vs HYBRID

----------------------------------------
1. LOADING DATA
----------------------------------------
Total ratings: 25,981,438


In [4]:
# ============================================================================
# RETRAIN ALS MODEL ON TEMPORAL SPLIT - MEMORY OPTIMIZED
# WITH PROPER MISSING MAPPING HANDLING
# ============================================================================

import numpy as np
import pandas as pd
import pickle
from scipy.sparse import csr_matrix, save_npz
import implicit
import time
import gc

print("=" * 60)
print("RETRAINING ALS MODEL ON TEMPORAL SPLIT - MEMORY OPTIMIZED")
print("=" * 60)

# ============================================================================
# 1. LOAD TEMPORAL TRAIN DATA WITH OPTIMAL DTYPES
# ============================================================================

print("\n" + "-" * 40)
print("Loading TEMPORAL train data with memory optimization...")
print("-" * 40)

# Define optimal dtypes to reduce memory
dtypes = {
    'userId': 'int32',
    'tmdbId': 'int32',
    'rating': 'float32',
    'timestamp': 'int32'
}

# Load with low_memory=False and optimal types
train_df = pd.read_csv(
    '/content/drive/MyDrive/ratings_train_vectorized.csv',
    dtype=dtypes,
    usecols=['userId', 'tmdbId', 'rating'],  # Only load what we need
    low_memory=False
)

print(f"Temporal train set: {len(train_df):,} ratings")
print(f"Memory usage: {train_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Get unique counts without loading full data
n_users_train = train_df['userId'].nunique()
n_movies_train = train_df['tmdbId'].nunique()
print(f"Users in train: {n_users_train:,}")
print(f"Movies in train: {n_movies_train:,}")

# Load mappers
print("\nLoading mappers...")
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)
with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)
with open('/content/drive/MyDrive/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)

print(f"User mapper: {len(user_mapper):,}")
print(f"Movie mapper: {len(movie_mapper):,}")

# ============================================================================
# 2. PROCESS IN CHUNKS TO BUILD INDICES AND CONFIDENCE
# ============================================================================

print("\n" + "-" * 40)
print("Processing data in chunks with missing mapping handling...")
print("-" * 40)

# Process in chunks to avoid memory explosion
chunk_size = 2_000_000
n_chunks = (len(train_df) + chunk_size - 1) // chunk_size

all_rows = []
all_cols = []
all_data = []
total_missing = 0
alpha = 40

for i, start in enumerate(range(0, len(train_df), chunk_size)):
    end = min(start + chunk_size, len(train_df))
    chunk = train_df.iloc[start:end].copy()

    print(f"\n  Processing chunk {i+1}/{n_chunks}: {len(chunk):,} rows")

    # STEP 1: Map first (creates NaNs for missing IDs)
    mapped_user = chunk['userId'].map(user_mapper)
    mapped_item = chunk['tmdbId'].map(movie_mapper)

    # STEP 2: Add to chunk temporarily
    chunk['user_idx'] = mapped_user
    chunk['item_idx'] = mapped_item

    # STEP 3: Count missing before dropping
    missing_before = len(chunk)

    # STEP 4: DROP THE MISSING MAPPINGS FIRST (CRITICAL FIX)
    chunk = chunk.dropna(subset=['user_idx', 'item_idx'])

    missing_after = len(chunk)
    missing_in_chunk = missing_before - missing_after
    total_missing += missing_in_chunk

    if missing_in_chunk > 0:
        print(f"     Dropped {missing_in_chunk:,} rows with missing mappings")

    if len(chunk) == 0:
        continue

    # STEP 5: NOW it is safe to cast to int32 (no NaNs remain)
    chunk['user_idx'] = chunk['user_idx'].astype('int32')
    chunk['item_idx'] = chunk['item_idx'].astype('int32')

    # Calculate confidence
    chunk['confidence'] = 1 + alpha * (chunk['rating'] / 5.0)

    # Store indices and values
    all_rows.extend(chunk['item_idx'].values)
    all_cols.extend(chunk['user_idx'].values)
    all_data.extend(chunk['confidence'].values.astype('float32'))

    # Clear chunk
    del chunk
    gc.collect()

print(f"\n✅ Total missing mappings dropped: {total_missing:,} rows")

# Free the original DataFrame
del train_df
gc.collect()

print(f"\nTotal interactions to build: {len(all_data):,}")

# ============================================================================
# 3. BUILD SPARSE MATRIX (FULL SIZE)
# ============================================================================

print("\n" + "-" * 40)
print("Building sparse matrix...")
print("-" * 40)

n_users = len(user_mapper)
n_items = len(movie_mapper)

# Convert to numpy arrays
rows = np.array(all_rows, dtype=np.int32)
cols = np.array(all_cols, dtype=np.int32)
data = np.array(all_data, dtype=np.float32)

# Clear lists to free memory
del all_rows, all_cols, all_data
gc.collect()

# Create sparse matrix with FULL dimensions
C_temporal = csr_matrix(
    (data, (rows, cols)),
    shape=(n_items, n_users),
    dtype=np.float32
)

print(f"Temporal training matrix shape: {C_temporal.shape}")
print(f"Non-zero entries: {C_temporal.nnz:,}")
print(f"Density: {C_temporal.nnz / (n_items * n_users) * 100:.6f}%")

# Free numpy arrays
del rows, cols, data
gc.collect()

# ============================================================================
# 4. TRAIN ALS MODEL ON TEMPORAL DATA (WITH TRANSPOSE)
# ============================================================================

print("\n" + "-" * 40)
print("Training ALS model on TEMPORAL data...")
print("-" * 40)

# CRITICAL: Transpose so users are rows (what ALS expects)
C_users_items = C_temporal.T.tocsr()

model_temporal = implicit.als.AlternatingLeastSquares(
    factors=100,
    regularization=0.01,
    iterations=50,
    random_state=42,
    num_threads=0,
    use_gpu=False
)

start_time = time.time()
model_temporal.fit(C_users_items, show_progress=True)
train_time = time.time() - start_time

print(f"\nTraining completed in {train_time:.2f} seconds")
print(f"User factors: {model_temporal.user_factors.shape}")  # Should be (n_users, 100)
print(f"Item factors: {model_temporal.item_factors.shape}")  # Should be (n_items, 100)

# ============================================================================
# 5. SAVE TEMPORAL MODEL
# ============================================================================

print("\n" + "-" * 40)
print("Saving temporal ALS model...")
print("-" * 40)

temporal_model_path = "/content/drive/MyDrive/als_model_temporal.pkl"
with open(temporal_model_path, "wb") as f:
    pickle.dump(model_temporal, f)

save_npz('/content/drive/MyDrive/als_confidence_matrix_temporal.npz', C_temporal)

print(f"✅ Temporal model saved to: {temporal_model_path}")
print(f"✅ Temporal confidence matrix saved")

# ============================================================================
# 6. QUICK TEST ON A SAMPLE USER
# ============================================================================

print("\n" + "-" * 40)
print("Quick test on sample user...")
print("-" * 40)

# Load test data with minimal memory footprint
test_df = pd.read_csv(
    '/content/drive/MyDrive/ratings_test_vectorized.csv',
    dtype={'userId': 'int32'},
    usecols=['userId'],
    nrows=1
)
sample_user = test_df['userId'].iloc[0]
del test_df
gc.collect()

print(f"Sample user: {sample_user}")

if sample_user in user_mapper:
    user_idx = user_mapper[sample_user]
    user_interactions = C_temporal[:, user_idx].nnz

    print(f"  User index: {user_idx}")
    print(f"  Interactions in training: {user_interactions}")

    if user_interactions > 0:
        user_items = C_temporal[:, user_idx].T.tocsr()
        recommendations = model_temporal.recommend(
            user_idx,
            user_items,
            N=10,
            filter_already_liked_items=True
        )

        print(f"  Top 10 recommendations:")
        for idx, score in zip(recommendations[0], recommendations[1]):
            if idx in movie_inv_mapper:
                movie_id = movie_inv_mapper[idx]
                print(f"    Movie {movie_id}: score {score:.4f}")
    else:
        print("  User has no interactions in training matrix")
else:
    print(f"  User {sample_user} not in mapper")

print("\n" + "=" * 60)
print("TEMPORAL ALS MODEL TRAINING COMPLETE")
print("=" * 60)

RETRAINING ALS MODEL ON TEMPORAL SPLIT - MEMORY OPTIMIZED

----------------------------------------
Loading TEMPORAL train data with memory optimization...
----------------------------------------
Temporal train set: 20,680,187 ratings
Memory usage: 236.67 MB
Users in train: 265,786
Movies in train: 38,719

Loading mappers...
User mapper: 270,882
Movie mapper: 45,423

----------------------------------------
Processing data in chunks with missing mapping handling...
----------------------------------------

  Processing chunk 1/11: 2,000,000 rows
     Dropped 1 rows with missing mappings

  Processing chunk 2/11: 2,000,000 rows

  Processing chunk 3/11: 2,000,000 rows

  Processing chunk 4/11: 2,000,000 rows

  Processing chunk 5/11: 2,000,000 rows

  Processing chunk 6/11: 2,000,000 rows

  Processing chunk 7/11: 2,000,000 rows

  Processing chunk 8/11: 2,000,000 rows

  Processing chunk 9/11: 2,000,000 rows

  Processing chunk 10/11: 2,000,000 rows

  Processing chunk 11/11: 680,187 

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/50 [00:00<?, ?it/s]


Training completed in 859.09 seconds
User factors: (270882, 100)
Item factors: (45423, 100)

----------------------------------------
Saving temporal ALS model...
----------------------------------------
✅ Temporal model saved to: /content/drive/MyDrive/als_model_temporal.pkl
✅ Temporal confidence matrix saved

----------------------------------------
Quick test on sample user...
----------------------------------------
Sample user: 1
  User index: 0
  Interactions in training: 21
  Top 10 recommendations:
    Movie 27205: score 1.0735
    Movie 12405: score 1.0055
    Movie 10681: score 0.9755
    Movie 14160: score 0.9731
    Movie 489: score 0.9713
    Movie 1422: score 0.9696
    Movie 1124: score 0.9427
    Movie 49026: score 0.9375
    Movie 235: score 0.9355
    Movie 16869: score 0.9259

TEMPORAL ALS MODEL TRAINING COMPLETE


In [8]:
# VECTORIZED TEMPORAL SPLIT FUNCTION
def create_temporal_split_vectorized(ratings_df, test_size=0.2):
    """
    VECTORIZED temporal train/test split
    - Single pass through data
    - Uses cumcount() for O(N log N) performance
    - No Python loops over users
    """
    print("Creating VECTORIZED temporal split...")
    start_time = time.time()

    # Sort all ratings by timestamp once
    df_sorted = ratings_df.sort_values(['userId', 'timestamp']).copy()

    # Count ratings per user (vectorized)
    df_sorted['user_rating_count'] = df_sorted.groupby('userId')['userId'].transform('count')

    # Calculate position within each user's history (0-based)
    df_sorted['user_position'] = df_sorted.groupby('userId').cumcount()

    # Determine test cut position for each user
    df_sorted['test_cut'] = (df_sorted['user_rating_count'] * (1 - test_size)).astype(int)

    # Create train/test masks in one vectorized operation
    train_mask = df_sorted['user_position'] < df_sorted['test_cut']
    test_mask = (df_sorted['user_position'] >= df_sorted['test_cut']) & (df_sorted['user_rating_count'] >= 5)

    # Apply masks
    train_df = df_sorted[train_mask].drop(columns=['user_rating_count', 'user_position', 'test_cut']).copy()
    test_df = df_sorted[test_mask].drop(columns=['user_rating_count', 'user_position', 'test_cut']).copy()

    # Clean up
    del df_sorted
    gc.collect()

    elapsed = time.time() - start_time
    print(f"Split completed in {elapsed:.2f} seconds")
    print(f"Train set: {len(train_df):,} ratings")
    print(f"Test set: {len(test_df):,} ratings")
    print(f"Users in test: {test_df['userId'].nunique():,}")
    print(f"Users with no test ratings (cold start): {train_df['userId'].nunique() - test_df['userId'].nunique():,}")

    return train_df, test_df

# Check if vectorized split files exist
if os.path.exists('/content/drive/MyDrive/ratings_train_vectorized.csv') and os.path.exists('/content/drive/MyDrive/ratings_test_vectorized.csv'):
    print("Loading existing vectorized train/test split files...")
    ratings_train = pd.read_csv('/content/drive/MyDrive/ratings_train_vectorized.csv')
    ratings_test = pd.read_csv('/content/drive/MyDrive/ratings_test_vectorized.csv')
    print(f"Train set: {len(ratings_train):,} ratings")
    print(f"Test set: {len(ratings_test):,} ratings")
else:
    print("Vectorized split files not found. Creating new split...")
    ratings_train, ratings_test = create_temporal_split_vectorized(ratings)
    ratings_train.to_csv('/content/drive/MyDrive/ratings_train_vectorized.csv', index=False)
    ratings_test.to_csv('/content/drive/MyDrive/ratings_test_vectorized.csv', index=False)
    print("Vectorized split saved for future use.")

    # Free memory
    del ratings
    gc.collect()


print("")
print("2. LOADING MODELS")
print("")

# Load CBF models
print(" Loading CBF models...")
with open('/content/drive/MyDrive/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('/content/drive/MyDrive/svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)
index = faiss.read_index('/content/drive/MyDrive/faiss_index.bin')
with open('/content/drive/MyDrive/movie_id_to_idx.pkl', 'rb') as f:
    movie_id_to_idx = pickle.load(f)
with open('/content/drive/MyDrive/idx_to_movie_id.pkl', 'rb') as f:
    idx_to_movie_id = pickle.load(f)
movies_df = pd.read_pickle('/content/drive/MyDrive/movies_df.pkl')
print(" CBF models loaded")

# Load CF model
print(" Loading CF model...")
with open('/content/drive/MyDrive/als_model_temporal.pkl', 'rb') as f:
    als_model = pickle.load(f)
with open('/content/drive/MyDrive/user_mapper.pkl', 'rb') as f:
    user_mapper = pickle.load(f)
with open('/content/drive/MyDrive/movie_mapper.pkl', 'rb') as f:
    movie_mapper = pickle.load(f)
with open('/content/drive/MyDrive/movie_inv_mapper.pkl', 'rb') as f:
    movie_inv_mapper = pickle.load(f)
print(" CF model loaded")

# Load confidence matrix
C = load_npz('/content/drive/MyDrive/als_confidence_matrix_temporal.npz')
print(f" Confidence matrix loaded: {C.shape}")

# Load hybrid config
print(" Loading hybrid config...")
with open('/content/drive/MyDrive/hybrid_config.pkl', 'rb') as f:
    hybrid_config = pickle.load(f)
print(" Hybrid config loaded")


print("")
print("3. ID VERIFICATION")
print("")

# Sample a few test items and verify they exist in mappers
sample_user = list(ratings_test['userId'].unique())[0]
sample_test_items = ratings_test[ratings_test['userId'] == sample_user]['tmdbId'].head(5).tolist()
print(f"Sample test items for user {sample_user}: {sample_test_items}")

# Check if these IDs exist in movie_mapper
missing_in_mapper = []
for movie_id in sample_test_items:
    if movie_id not in movie_mapper:
        missing_in_mapper.append(movie_id)

if missing_in_mapper:
    print(f" WARNING: {len(missing_in_mapper)} test items not found in movie_mapper")
    print(f"Missing IDs: {missing_in_mapper}")

    # Load links.csv to check ID mapping
    if os.path.exists('/content/drive/MyDrive/links.csv'):
        links = pd.read_csv('/content/drive/MyDrive/links.csv')
        print("\nLinks.csv columns:", links.columns.tolist())
        for movie_id in missing_in_mapper:
            match = links[links['tmdbId'] == movie_id]
            if not match.empty:
                print(f"  Movie ID {movie_id} exists in links.csv (movieId: {match['movieId'].values[0]})")
else:
    print("✓ All test items found in movie_mapper - IDs are consistent")


print("")
print("4. SELECTING TEST USERS")
print("")

# Find users with at least 3 relevant items in test set
test_user_counts = ratings_test['userId'].value_counts()
potential_users = test_user_counts[test_user_counts >= 3].index.tolist()
print(f"Users with ≥3 test ratings: {len(potential_users)}")

# Sample 500 users
n_sample = min(500, len(potential_users))
random.seed(42)
test_users = random.sample(potential_users, n_sample)
print(f"Sampled {len(test_users)} users for evaluation")

# Get relevant items (rating ≥ 3.5) for each test user
user_relevant = {}
for user_id in test_users:
    user_test = ratings_test[ratings_test['userId'] == user_id]
    relevant = user_test[user_test['rating'] >= 3.5]['tmdbId'].tolist()
    if len(relevant) >= 3:
        user_relevant[user_id] = relevant

print(f"Final evaluation users: {len(user_relevant)} (with ≥3 relevant items)")

# Get user's train items (for CBF query seeding)
user_train_items = {}
for user_id in user_relevant.keys():
    train_items = ratings_train[ratings_train['userId'] == user_id]['tmdbId'].tolist()
    user_train_items[user_id] = train_items[:10]

del ratings_train
del ratings_test
gc.collect()



def precision_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]
    hits = len(set(recommended) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0

def ndcg_at_k(recommended, relevant, k):
    if len(recommended) > k:
        recommended = recommended[:k]

    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal = sum([1 / np.log2(i + 2) for i in range(min(len(relevant), k))])
    return dcg / ideal if ideal > 0 else 0

def map_at_k(recommended, relevant, k):
    """Mean Average Precision @ K"""
    if len(recommended) > k:
        recommended = recommended[:k]

    ap = 0
    hits = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k) if len(relevant) > 0 else 0


def get_cbf_recommendations_with_scores(user_train_movies, n=20):
    if not user_train_movies:
        return [], {}

    all_candidates = defaultdict(float)
    all_scores = {}

    for movie_id in user_train_movies[:5]:
        if movie_id not in movie_id_to_idx:
            continue

        movie_row = movies_df[movies_df['id'] == movie_id]
        if len(movie_row) == 0:
            continue

        movie_content = movie_row.iloc[0]['combined_content']

        tfidf_vec = tfidf.transform([movie_content])
        svd_vec = svd.transform(tfidf_vec).astype('float32')
        svd_vec = np.ascontiguousarray(svd_vec, dtype=np.float32)
        faiss.normalize_L2(svd_vec)

        distances, indices = index.search(svd_vec, 50)

        for i, idx in enumerate(indices[0][1:]):
            cand_id = idx_to_movie_id[idx]
            sim_score = float(distances[0][i+1])
            score = sim_score * (1.0 / (i+2))
            all_candidates[cand_id] += score
            all_scores[cand_id] = sim_score

    sorted_cands = sorted(all_candidates.items(), key=lambda x: x[1], reverse=True)
    recommendations = [c[0] for c in sorted_cands[:n]]
    score_dict = {mid: all_scores.get(mid, 0) for mid in recommendations}

    return recommendations, score_dict


def get_cf_recommendations(user_id, n=20):
    if user_id not in user_mapper:
        return [], {}

    user_idx = user_mapper[user_id]
    user_interactions = C[:, user_idx].nnz

    if user_interactions == 0:
        return [], {}

    user_items = C[:, user_idx].T.tocsr()

    try:
        recommendations = als_model.recommend(
            user_idx,
            user_items,
            N=n,
            filter_already_liked_items=True
        )

        rec_ids = []
        score_dict = {}
        for idx, score in zip(recommendations[0], recommendations[1]):
            if idx in movie_inv_mapper:
                movie_id = movie_inv_mapper[idx]
                rec_ids.append(movie_id)
                score_dict[movie_id] = float(score)

        return rec_ids, score_dict

    except Exception as e:
        print(f"Error for user {user_id}: {e}")
        return [], {}


def get_hybrid_recommendations_improved(user_id, user_train_movies, n=20):
    if not user_train_movies:
        return [], {}

    # 1. Fetch Candidates
    cf_recs, _ = get_cf_recommendations(user_id, n=200)
    if not cf_recs:
        return [], {}

    cbf_recs, _ = get_cbf_recommendations_with_scores(user_train_movies, n=200)

    # 2. Create rank dictionaries
    cf_rank = {movie_id: idx for idx, movie_id in enumerate(cf_recs)}
    cbf_rank = {movie_id: idx for idx, movie_id in enumerate(cbf_recs)}

    # 3. RRF Parameters - Introducing Weights
    k = 60
    w_cf = 1.0   # Primary signal
    w_cbf = 0.08  # Secondary signal (adjusted based on your 0.008 vs 0.087 precision)

    all_candidates = set(cf_recs) | set(cbf_recs)

    rrf_scores = {}
    for movie_id in all_candidates:
        score = 0
        if movie_id in cf_rank:
            score += w_cf / (k + cf_rank[movie_id])
        if movie_id in cbf_rank:
            score += w_cbf / (k + cbf_rank[movie_id])
        rrf_scores[movie_id] = score

    # 4. Final Sort
    sorted_hybrid = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    recommendations = [mid for mid, _ in sorted_hybrid[:n]]

    return recommendations, {mid: rrf_scores[mid] for mid in recommendations}

print("")
print("9. RUNNING EVALUATION WITH IMPROVED HYBRID")
print("")

# Initialize results dictionary with all metrics
results = {
    'cbf': {'p@5': [], 'r@5': [], 'ndcg@5': [], 'map@5': [],
            'p@10': [], 'r@10': [], 'ndcg@10': [], 'map@10': [],
            'p@20': [], 'r@20': [], 'ndcg@20': [], 'map@20': []},
    'cf': {'p@5': [], 'r@5': [], 'ndcg@5': [], 'map@5': [],
           'p@10': [], 'r@10': [], 'ndcg@10': [], 'map@10': [],
           'p@20': [], 'r@20': [], 'ndcg@20': [], 'map@20': []},
    'hybrid_improved': {'p@5': [], 'r@5': [], 'ndcg@5': [], 'map@5': [],
                        'p@10': [], 'r@10': [], 'ndcg@10': [], 'map@10': [],
                        'p@20': [], 'r@20': [], 'ndcg@20': [], 'map@20': []}
}

valid_users = []
cf_failures = 0
cf_short = 0

for i, user_id in enumerate(list(user_relevant.keys())):
    if i % 25 == 0 and i > 0:
        print(f"  Processed {i}/{len(user_relevant)} users")

    relevant = user_relevant[user_id]
    train_movies = user_train_items.get(user_id, [])

    if len(train_movies) < 3:
        continue

    cbf_recs, _ = get_cbf_recommendations_with_scores(train_movies, 20)
    cf_recs, _ = get_cf_recommendations(user_id, 20)
    hybrid_improved_recs, _ = get_hybrid_recommendations_improved(user_id, train_movies, 20)

    if len(cf_recs) == 0:
        cf_failures += 1
        continue

    if len(cf_recs) < 20:
        cf_short += 1

    if len(cbf_recs) >= 20 and len(cf_recs) >= 20 and len(hybrid_improved_recs) >= 20:
        valid_users.append(user_id)

        for k in [5, 10, 20]:
            # CBF metrics
            results['cbf'][f'p@{k}'].append(precision_at_k(cbf_recs, relevant, k))
            results['cbf'][f'r@{k}'].append(recall_at_k(cbf_recs, relevant, k))
            results['cbf'][f'ndcg@{k}'].append(ndcg_at_k(cbf_recs, relevant, k))
            results['cbf'][f'map@{k}'].append(map_at_k(cbf_recs, relevant, k))

            # CF metrics
            results['cf'][f'p@{k}'].append(precision_at_k(cf_recs, relevant, k))
            results['cf'][f'r@{k}'].append(recall_at_k(cf_recs, relevant, k))
            results['cf'][f'ndcg@{k}'].append(ndcg_at_k(cf_recs, relevant, k))
            results['cf'][f'map@{k}'].append(map_at_k(cf_recs, relevant, k))

            # Improved Hybrid metrics
            results['hybrid_improved'][f'p@{k}'].append(precision_at_k(hybrid_improved_recs, relevant, k))
            results['hybrid_improved'][f'r@{k}'].append(recall_at_k(hybrid_improved_recs, relevant, k))
            results['hybrid_improved'][f'ndcg@{k}'].append(ndcg_at_k(hybrid_improved_recs, relevant, k))
            results['hybrid_improved'][f'map@{k}'].append(map_at_k(hybrid_improved_recs, relevant, k))

print("")
print(f"CF Debug Summary:")
print(f"  Total users: {len(user_relevant)}")
print(f"  CF failures (0 items): {cf_failures}")
print(f"  CF short (<20 items): {cf_short}")
print(f"  Valid users (all models ≥20): {len(valid_users)}")


print("")
print("FINAL COMPARATIVE RESULTS - IMPROVED HYBRID")
print("")
print(f"(Based on {len(valid_users)} users with all models succeeding)")

for k in [5, 10, 20]:
    print("")
    print(f" @{k} METRICS")
    print("")

    print(f"Precision@{k}:")
    print(f"   CBF:              {np.mean(results['cbf'][f'p@{k}']):.4f} ± {np.std(results['cbf'][f'p@{k}']):.4f}")
    print(f"   CF:               {np.mean(results['cf'][f'p@{k}']):.4f} ± {np.std(results['cf'][f'p@{k}']):.4f}")
    print(f"   Hybrid Improved:  {np.mean(results['hybrid_improved'][f'p@{k}']):.4f} ± {np.std(results['hybrid_improved'][f'p@{k}']):.4f}")

    # Calculate improvement
    if np.mean(results['cbf'][f'p@{k}']) > 0:
        impr_vs_cbf = (np.mean(results['hybrid_improved'][f'p@{k}']) - np.mean(results['cbf'][f'p@{k}'])) / np.mean(results['cbf'][f'p@{k}']) * 100
        print(f"   Improvement vs CBF: +{impr_vs_cbf:.1f}%")

    print(f"\nRecall@{k}:")
    print(f"   CBF:              {np.mean(results['cbf'][f'r@{k}']):.4f} ± {np.std(results['cbf'][f'r@{k}']):.4f}")
    print(f"   CF:               {np.mean(results['cf'][f'r@{k}']):.4f} ± {np.std(results['cf'][f'r@{k}']):.4f}")
    print(f"   Hybrid Improved:  {np.mean(results['hybrid_improved'][f'r@{k}']):.4f} ± {np.std(results['hybrid_improved'][f'r@{k}']):.4f}")

    print(f"\nNDCG@{k}:")
    print(f"   CBF:              {np.mean(results['cbf'][f'ndcg@{k}']):.4f} ± {np.std(results['cbf'][f'ndcg@{k}']):.4f}")
    print(f"   CF:               {np.mean(results['cf'][f'ndcg@{k}']):.4f} ± {np.std(results['cf'][f'ndcg@{k}']):.4f}")
    print(f"   Hybrid Improved:  {np.mean(results['hybrid_improved'][f'ndcg@{k}']):.4f} ± {np.std(results['hybrid_improved'][f'ndcg@{k}']):.4f}")

    print(f"\nMAP@{k}:")
    print(f"   CBF:              {np.mean(results['cbf'][f'map@{k}']):.4f} ± {np.std(results['cbf'][f'map@{k}']):.4f}")
    print(f"   CF:               {np.mean(results['cf'][f'map@{k}']):.4f} ± {np.std(results['cf'][f'map@{k}']):.4f}")
    print(f"   Hybrid Improved:  {np.mean(results['hybrid_improved'][f'map@{k}']):.4f} ± {np.std(results['hybrid_improved'][f'map@{k}']):.4f}")
    print("")


Loading existing vectorized train/test split files...
Train set: 20,680,187 ratings
Test set: 5,286,420 ratings

2. LOADING MODELS

 Loading CBF models...
 CBF models loaded
 Loading CF model...
 CF model loaded
 Confidence matrix loaded: (45423, 270882)
 Loading hybrid config...
 Hybrid config loaded

3. ID VERIFICATION

Sample test items for user 1: [77, 10474, 58574, 49051, 70160]
✓ All test items found in movie_mapper - IDs are consistent

4. SELECTING TEST USERS

Users with ≥3 test ratings: 227044
Sampled 500 users for evaluation
Final evaluation users: 395 (with ≥3 relevant items)

9. RUNNING EVALUATION WITH IMPROVED HYBRID

  Processed 25/395 users
  Processed 50/395 users
  Processed 75/395 users
  Processed 100/395 users
  Processed 125/395 users
  Processed 150/395 users
  Processed 175/395 users
  Processed 200/395 users
  Processed 225/395 users
  Processed 250/395 users
  Processed 275/395 users
  Processed 300/395 users
  Processed 325/395 users
  Processed 350/395 users


In [10]:
# ============================================================================
# 11. LOG TO MLFLOW (DAGSHUB) - DIRECT CREDENTIAL INPUT
# ============================================================================

print("")
print("11. LOGGING TO MLFLOW (DAGSHUB)")
print("")

# DIRECT CREDENTIAL INPUT - Replace with your actual credentials
# You can get these from: https://dagshub.com -> Settings -> Personal Access Tokens
MLFLOW_TRACKING_USERNAME = "ayanfeoluwadegoke"  # ← REPLACE WITH YOUR USERNAME
MLFLOW_TRACKING_PASSWORD = "fda68f4b027ac36db2590129feed5debca12989b"      # ← REPLACE WITH YOUR TOKEN
REPO_NAME = "ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System"           # ← REPLACE WITH YOUR REPO (e.g., "ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System")

# OR use interactive input (safer - no hardcoded credentials):
# MLFLOW_TRACKING_USERNAME = input("Enter your DagsHub username: ").strip()
# MLFLOW_TRACKING_PASSWORD = getpass.getpass("Enter your DagsHub token: ").strip()
# REPO_NAME = input("Enter your DagsHub repo (username/repo_name): ").strip()

if MLFLOW_TRACKING_USERNAME and MLFLOW_TRACKING_PASSWORD and REPO_NAME and len(valid_users) > 0:
    # Set credentials in environment
    os.environ['MLFLOW_TRACKING_USERNAME'] = MLFLOW_TRACKING_USERNAME
    os.environ['MLFLOW_TRACKING_PASSWORD'] = MLFLOW_TRACKING_PASSWORD

    # Clean repo name and set tracking URI
    CLEAN_REPO = REPO_NAME.strip("/")
    DAGSHUB_TRACKING_URI = f"https://dagshub.com/{CLEAN_REPO}.mlflow"
    mlflow.set_tracking_uri(DAGSHUB_TRACKING_URI)

    print(f"✅ Authenticated as: {MLFLOW_TRACKING_USERNAME}")
    print(f"✅ Tracking to: {DAGSHUB_TRACKING_URI}")

    # Set experiment
    EXPERIMENT_NAME = "Movie_Recommender_Hybrid_Improved"
    try:
        mlflow.set_experiment(EXPERIMENT_NAME)
        print(f"✅ Using experiment: {EXPERIMENT_NAME}")
    except Exception as e:
        print(f"⚠️ Experiment warning: {e}")

    # Start run
    with mlflow.start_run(run_name=f"hybrid_improved_{datetime.now().strftime('%Y%m%d_%H%M')}") as run:

        run_id = run.info.run_id
        run_url = f"https://dagshub.com/{CLEAN_REPO}/experiments/#/experiments/{run.info.experiment_id}/runs/{run_id}"
        print(f"\n✅ Run started:")
        print(f"   Run ID: {run_id}")
        print(f"   Run URL: {run_url}")

        # Log parameters
        mlflow.log_param("n_users", len(valid_users))
        mlflow.log_param("threshold", 3.5)
        mlflow.log_param("split_type", "temporal_vectorized")
        mlflow.log_param("candidate_pool", 500)
        mlflow.log_param("cf_normalization", "tanh_scaling")

        # Log metrics for all K values
        for k in [5, 10, 20]:
            for model in ['cbf', 'cf', 'hybrid_improved']:
                if len(results[model][f'p@{k}']) > 0:
                    mlflow.log_metric(f"{model}_precision_{k}", float(np.mean(results[model][f'p@{k}'])))
                    mlflow.log_metric(f"{model}_recall_{k}", float(np.mean(results[model][f'r@{k}'])))
                    mlflow.log_metric(f"{model}_ndcg_{k}", float(np.mean(results[model][f'ndcg@{k}'])))
                    mlflow.log_metric(f"{model}_map_{k}", float(np.mean(results[model][f'map@{k}'])))

        # Log improvement percentages
        for k in [5, 10, 20]:
            if np.mean(results['cbf'][f'p@{k}']) > 0:
                impr_vs_cbf = (np.mean(results['hybrid_improved'][f'p@{k}']) - np.mean(results['cbf'][f'p@{k}'])) / np.mean(results['cbf'][f'p@{k}']) * 100
                mlflow.log_metric(f"hybrid_improvement_vs_cbf_pct_{k}", float(impr_vs_cbf))

        # Log tags
        mlflow.set_tag("evaluation_type", "comparative_temporal")
        mlflow.set_tag("cbf_model", "tfidf+svd+faiss")
        mlflow.set_tag("cf_model", "als")
        mlflow.set_tag("hybrid_fusion", "improved_tanh_scaling")
        mlflow.set_tag("split_method", "vectorized_cumcount")
        mlflow.set_tag("status", "production_ready")
        mlflow.set_tag("timestamp", datetime.now().isoformat())

        print("\n✅ All metrics logged successfully to DagsHub")
        print(f"📊 View your run at: {run_url}")

else:
    print("⚠️ Missing credentials or no valid users, skipping MLflow logging")
    print("To log to DagsHub, set your credentials in the section above")

print("")
print("IMPROVED HYBRID EVALUATION COMPLETE")
print("")


11. LOGGING TO MLFLOW (DAGSHUB)

✅ Authenticated as: ayanfeoluwadegoke
✅ Tracking to: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System.mlflow
✅ Using experiment: Movie_Recommender_Hybrid_Improved

✅ Run started:
   Run ID: 0ada17d029354f189dea0efd004cb513
   Run URL: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System/experiments/#/experiments/3/runs/0ada17d029354f189dea0efd004cb513

✅ All metrics logged successfully to DagsHub
📊 View your run at: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System/experiments/#/experiments/3/runs/0ada17d029354f189dea0efd004cb513
🏃 View run hybrid_improved_20260303_1654 at: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System.mlflow/#/experiments/3/runs/0ada17d029354f189dea0efd004cb513
🧪 View experiment at: https://dagshub.com/ayanfeoluwadegoke/Hybrid_Movie_Recommendation_System.mlflow/#/experiments/3

IMPROVED HYBRID EVALUATION COMPLETE



In [11]:
# ============================================================================
# 12. LOG ALL NEW ARTIFACTS TO MLFLOW
# ============================================================================

print("")
print("12. LOGGING NEW ARTIFACTS TO MLFLOW")
print("")

# List of new artifacts to log
new_artifacts = [
    # Temporal ALS model and matrix
    ("/content/drive/MyDrive/als_model_temporal.pkl", "models/cf/als_model_temporal"),
    ("/content/drive/MyDrive/als_confidence_matrix_temporal.npz", "matrices/als_confidence_temporal"),

    # Updated hybrid config
    ("/content/drive/MyDrive/hybrid_config.pkl", "models/hybrid/hybrid_config_final"),

    # # Evaluation results
    # ("/content/drive/MyDrive/ratings_train_vectorized.csv", "processed_data/ratings_train_temporal"),
    # ("/content/drive/MyDrive/ratings_test_vectorized.csv", "processed_data/ratings_test_temporal"),
]

# Function to log a single artifact
def log_artifact_to_mlflow(file_path, artifact_path):
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path) / (1024 * 1024)
        mlflow.log_artifact(file_path, artifact_path)
        print(f"  ✅ Logged: {os.path.basename(file_path)} ({file_size:.2f} MB) → {artifact_path}")
        return True
    else:
        print(f"  ⚠ File not found: {file_path}")
        return False

# Log each artifact within the existing MLflow run
for file_path, artifact_path in new_artifacts:
    log_artifact_to_mlflow(file_path, artifact_path)

print("")
print("✅ All new artifacts logged successfully")


12. LOGGING NEW ARTIFACTS TO MLFLOW

  ✅ Logged: als_model_temporal.pkl (120.66 MB) → models/cf/als_model_temporal
  ✅ Logged: als_confidence_matrix_temporal.npz (46.99 MB) → matrices/als_confidence_temporal
  ✅ Logged: hybrid_config.pkl (0.56 MB) → models/hybrid/hybrid_config_final

✅ All new artifacts logged successfully
